In [ ]:
# 1. Setup de caminhos locais
from datetime import datetime

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "05_data_collection_real"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


In [ ]:
# 3. Buscar notícias (NewsAPI) com período parametrizado
import requests
import pandas as pd
from datetime import datetime, timedelta
import os

# SECURITY: Get API key from environment variable
API_KEY = os.environ.get("NEWSAPI_KEY")
if not API_KEY:
    print("❌ ERRO: Configure a variável de ambiente NEWSAPI_KEY")
    print("   Windows CMD: set NEWSAPI_KEY=sua_chave_aqui")
    print("   PowerShell:  $env:NEWSAPI_KEY='sua_chave_aqui'")
    print("   Linux/Mac:   export NEWSAPI_KEY=sua_chave_aqui")
    raise ValueError("NEWSAPI_KEY não configurada")

# Get date range from config
periodo = cfg.get_periodo_estudo()
start_date = pd.to_datetime(periodo["start"])
end_date = pd.to_datetime(periodo["end"])

print(f"📅 Período de coleta: {start_date.date()} → {end_date.date()}")

URL = "https://newsapi.org/v2/everything"

# NewsAPI limitation: max 1 month lookback for free tier
# We'll collect recent news as a demonstration
# For full historical data, you need a paid plan or alternative sources
lookback_days = min(30, (datetime.now() - start_date).days)
collect_from = datetime.now() - timedelta(days=lookback_days)

print(f"⚠️ NewsAPI free tier limitado a ~30 dias. Coletando de {collect_from.date()}")
print(f"⚠️ Para dados históricos completos {start_date.date()}→{end_date.date()}, use plano pago ou fontes alternativas")

params = {
    "q": "Ibovespa OR Bovespa OR Petrobras OR Vale OR dólar OR economia",
    "language": "pt",
    "sortBy": "publishedAt",
    "from": collect_from.strftime("%Y-%m-%d"),
    "pageSize": 100,
    "apiKey": API_KEY
}

# Pagination support: collect multiple pages
all_articles = []
for page in range(1, 4):  # Collect up to 3 pages (300 articles)
    params["page"] = page
    resp = requests.get(URL, params=params)
    
    if resp.status_code != 200:
        print(f"⚠️ Erro {resp.status_code} na página {page}: {resp.text[:200]}")
        break
    
    data = resp.json()
    articles = data.get("articles", [])
    
    if not articles:
        print(f"✓ Página {page}: sem mais artigos")
        break
    
    print(f"✓ Página {page}: {len(articles)} artigos coletados")
    
    for art in articles:
        all_articles.append({
            "data": art["publishedAt"][:10],  # formato YYYY-MM-DD
            "fonte": art["source"]["name"],
            "titulo": art["title"],
            "texto": art["description"] or ""
        })

df_new = pd.DataFrame(all_articles)
print(f"\n📊 Total de novas notícias coletadas: {df_new.shape[0]}")

if df_new.empty:
    print("❌ ERRO: Nenhuma notícia coletada. Verifique:")
    print("   - API key válida")
    print("   - Limite de requisições não excedido")
    print("   - Conectividade com a API")
    raise ValueError("Coleta retornou 0 notícias - pipeline interrompido")

display(df_new.head())

# 4. Incrementar dataset (sem fallback sintético)
out_file = os.path.join(RAW_PATH, "noticias_real.csv")

if os.path.exists(out_file):
    df_old = pd.read_csv(out_file)
    print(f"Dataset anterior: {df_old.shape[0]} notícias")
else:
    df_old = pd.DataFrame(columns=["data","fonte","titulo","texto"])
    print("Nenhum dataset anterior encontrado, iniciando novo arquivo.")

# Concatenar e remover duplicatas
df_full = pd.concat([df_old, df_new], ignore_index=True)
df_full = df_full.drop_duplicates(subset=["data","titulo"]).reset_index(drop=True)

# Salvar atualizado
df_full.to_csv(out_file, index=False, encoding="utf-8-sig")

print(f"\n✅ Atualização concluída!")
print(f"   Total de notícias: {df_full.shape[0]}")
print(f"   Período: {df_full['data'].min()} → {df_full['data'].max()}")
print(f"   Arquivo: {out_file}")


In [ ]:
# Validar período de cobertura
PERIODO = cfg.get_periodo_estudo()
START_EXPECTED = pd.to_datetime(PERIODO["start"])
END_EXPECTED = pd.to_datetime(PERIODO["end"])

if "data" in df_all.columns:
    min_date = pd.to_datetime(df_all["data"].min())
    max_date = pd.to_datetime(df_all["data"].max())
    print(f"\n=== Validação de Período ===")
    print(f"Esperado: {START_EXPECTED.date()} → {END_EXPECTED.date()}")
    print(f"Obtido:   {min_date.date()} → {max_date.date()}")
    
    if min_date > START_EXPECTED + pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Início dos dados ({min_date.date()}) está >30 dias após o esperado.")
    if max_date < END_EXPECTED - pd.Timedelta(days=30):
        print(f"⚠️ AVISO: Fim dos dados ({max_date.date()}) está >30 dias antes do esperado.")
    if min_date <= START_EXPECTED and max_date >= END_EXPECTED:
        print("✅ Período validado com sucesso!")